# S_CHIP2CHIP

In [1]:

from pynq import (PL, allocate, Overlay)
import numpy as np
from PIL import Image
from datetime import datetime

PL.reset()
ol = Overlay("slave-zcu102.bit")


In [2]:
import time

VDMA_S2MM = {
    "S2MM_VDMACR": 0x30,
    "S2MM_VDMASR": 0x34,
    "S2MM_VDMA_IRQ_MASK": 0x3C,
    "S2MM_REG_INDEX": 0x44,
    "S2MM_VSIZE": 0xA0,
    "S2MM_HSIZE": 0xA4,
    "S2MM_STRIDE": 0xA8,
    "S2MM_SA1": 0xAC,
    "S2MM_SA2": 0xB0,
    "S2MM_SA3": 0xB4,
    "S2MM_SA4": 0xB8,
    "S2MM_SA5": 0xBC,
    "S2MM_SA6": 0xC0,
    "S2MM_SA7": 0xC4,
    "S2MM_SA8": 0xC8,
    "S2MM_SA9": 0xCC,
    "S2MM_SA10": 0xD0,
    "S2MM_SA11": 0xD4,
    "S2MM_SA12": 0xD8,
    "S2MM_SA13": 0xDC,
    "S2MM_SA14": 0xE0,
    "S2MM_SA15": 0xE4,
    "S2MM_SA16": 0xE8,
}

def vdma_readframes(_vdma, _stride, _v_size, _frame_rcv1, _frame_rcv2):
    # ----------------------------------------
    # Reset S2MM Channel
    # ----------------------------------------
    _vdma.write(VDMA_S2MM["S2MM_VDMACR"], 0x00000004)  # Reset
    time.sleep(0.01)
    _vdma.write(VDMA_S2MM["S2MM_VDMACR"], 0x00000001)  # Run/Stop = 1, circular mode = 0
    #_vdma.write(VDMA_S2MM["S2MM_VDMACR"], 0x00000003)  # Run/Stop = 1, circular mode = 1

    # ----------------------------------------
    # Set frame buffer base addresses
    # ----------------------------------------
    _vdma.write(VDMA_S2MM["S2MM_REG_INDEX"], 0x0)
    _vdma.write(VDMA_S2MM["S2MM_SA1"], _frame_rcv1.physical_address)
    _vdma.write(VDMA_S2MM["S2MM_SA2"], _frame_rcv2.physical_address)

    # ----------------------------------------
    # Set stride (bytes per row)
    # ----------------------------------------
    _vdma.write(VDMA_S2MM["S2MM_STRIDE"], _stride)

    # ----------------------------------------
    # Set horizontal size (in bytes)
    # ----------------------------------------
    _vdma.write(VDMA_S2MM["S2MM_HSIZE"], _stride)

    # ----------------------------------------
    # Set vertical size (in lines) to trigger transfer
    # ----------------------------------------
    _vdma.write(VDMA_S2MM["S2MM_VSIZE"], _v_size)

def dump_s2mm_status(_vdma):
    # Read relevant registers using map
    cr = _vdma.read(VDMA_S2MM["S2MM_VDMACR"])
    sr = _vdma.read(VDMA_S2MM["S2MM_VDMASR"])
    vsize = _vdma.read(VDMA_S2MM["S2MM_VSIZE"])
    hsize = _vdma.read(VDMA_S2MM["S2MM_HSIZE"])
    stride = _vdma.read(VDMA_S2MM["S2MM_STRIDE"])
    sa1 = _vdma.read(VDMA_S2MM["S2MM_SA1"])
    sa2 = _vdma.read(VDMA_S2MM["S2MM_SA2"])

    print("----- VDMA S2MM Status Dump -----")
    print(f"Control Reg     (0x{VDMA_S2MM['S2MM_VDMACR']:02X}): 0x{cr:08X}")
    print(f"Status Reg      (0x{VDMA_S2MM['S2MM_VDMASR']:02X}): 0x{sr:08X}")
    print(f"Vertical Size   (0x{VDMA_S2MM['S2MM_VSIZE']:02X}): {vsize}")
    print(f"Horizontal Size (0x{VDMA_S2MM['S2MM_HSIZE']:02X}): {hsize}")
    print(f"Stride          (0x{VDMA_S2MM['S2MM_STRIDE']:02X}): {stride}")
    print(f"S2MM_SA1        (0x{VDMA_S2MM['S2MM_SA1']:02X}): 0x{sa1:08X}")
    print(f"S2MM_SA2        (0x{VDMA_S2MM['S2MM_SA2']:02X}): 0x{sa2:08X}")

    # Decode common status bits (optional)
    status_bits = {
        0:  "HALTED",
        1:  "VDMA Internal Error",
        2:  "Slave Error",
        3:  "Decode Error",
        4:  "Start of Frame Early Error",
        5:  "End of Line Early Error",
        6:  "Start of Frame Late Error",
        10: "End of Line Late Error",
        12: "Frame Count Interrupt",
        13: "Delay Count Interrupt",
        14: "Error Interrupt",
        31: "DMA Internal Halted"
    }

    print("Status Flags:")
    for bit, description in status_bits.items():
        if sr & (1 << bit):
            print(f" - Bit {bit}: {description}")

    print("----------------------------------")



In [3]:
def dump_hls_ctrl(hls_mmio):
    ADDR_AP_CTRL            =0x00
    ADDR_GIE                =0x04
    ADDR_IER                =0x08
    ADDR_ISR                =0x0c

    # Read relevant registers
    ctrl = hls_mmio.read(ADDR_AP_CTRL)
    gie = hls_mmio.read(ADDR_GIE)
    ier = hls_mmio.read(ADDR_IER)
    isr = hls_mmio.read(ADDR_ISR)

    print("----- hls Dump -----")
    print(f"Control signals                  0x{ADDR_AP_CTRL:02X}: 0x{ctrl:08X}")
    print(f"Global Interrupt Enable Register 0x{ADDR_GIE:02X}: 0x{gie:08X}")
    print(f"IP Interrupt Enable Register     0x{ADDR_IER:02X}: 0x{ier:08X}")
    print(f"IP Interrupt Status Register     0x{ADDR_ISR:02X}: 0x{isr:08X}")
    

    # Decode common status bits (optional)
    status_bits = {
        0:  "ap_start (Read/Write/COH)",
        1:  "ap_done (Read/COR)",
        2:  "ap_idle (Read)",
        3:  "ap_ready (Read/COR)",
        7:  "auto_restart (Read/Write)",
        9:  "interrupt (Read)",
    }

    print("Control Flags:")
    for bit, description in status_bits.items():
        if ctrl & (1 << bit):
            print(f" - Bit {bit}: {description}")

    print("----------------------------------")

def img_to_axis(ip_mmio,buffer, eos,frame_cnt,loop):
    ADDR_AP_CTRL            =0x00
    ADDR_GIE                =0x04
    ADDR_IER                =0x08
    ADDR_ISR                =0x0c
    ADDR_DATA_PORT_DATA     =0x10
    BITS_DATA_PORT_DATA     =32
    ADDR_FRAME_CNT_DATA     =0x18
    BITS_FRAME_CNT_DATA     =32
    ADDR_END_OF_STREAM_DATA =0x20
    BITS_END_OF_STREAM_DATA =1
    
# Configure registers:
    # Write physical address of buffer to data_port register
    ip_mmio.write(ADDR_DATA_PORT_DATA , buffer.physical_address)
    # Set end_of_stream 
    ip_mmio.write(ADDR_END_OF_STREAM_DATA , eos)
    # Set frame_no to 88
    ip_mmio.write(ADDR_FRAME_CNT_DATA , frame_cnt)
    # Start the IP core by setting the ap_start bit in CTRL register
    if loop:
        ip_mmio.write(ADDR_AP_CTRL , (0x1<<7 |0x1))
    else:
        ip_mmio.write(ADDR_AP_CTRL , 0x1)

def cfg_img_to_axis(ip_mmio,buffer, eos,frame_cnt):
    ADDR_AP_CTRL            =0x00
    ADDR_GIE                =0x04
    ADDR_IER                =0x08
    ADDR_ISR                =0x0c
    ADDR_DATA_PORT_DATA     =0x10
    BITS_DATA_PORT_DATA     =32
    ADDR_FRAME_CNT_DATA     =0x18
    BITS_FRAME_CNT_DATA     =32
    ADDR_END_OF_STREAM_DATA =0x20
    BITS_END_OF_STREAM_DATA =1
    
# Configure registers:
    # Write physical address of buffer to data_port register
    ip_mmio.write(ADDR_DATA_PORT_DATA , buffer.physical_address)
    # Set end_of_stream 
    ip_mmio.write(ADDR_END_OF_STREAM_DATA , eos)
    # Set frame_no to 88
    ip_mmio.write(ADDR_FRAME_CNT_DATA , frame_cnt)


In [4]:
def image_to_RGB(image_fname):
    # === LOAD AND CONVERT IMAGE TO RGB ===
    img = Image.open(f"{image_fname}").convert('RGB') 
    img_np = np.array(img)  # Shape: (H, W, 3), dtype=uint8

    # === PACK RGB TO INT32 ===
    # Format: 0x00RRGGBB (most significant byte can be 0)
    r = img_np[:, :, 0].astype(np.uint32)
    g = img_np[:, :, 1].astype(np.uint32)
    b = img_np[:, :, 2].astype(np.uint32)
    rgb_packed = (b << 16) | (g << 8) | r  # Shape: (H, W)

    # Allocate contiguous buffer with dtype uint32
    buffer = allocate(shape=rgb_packed.shape, dtype=np.uint32)

    # Copy packed pixels into buffer
    np.copyto(buffer, rgb_packed)

    print(f"Packed buffer shape: {buffer.shape}, dtype: {buffer.dtype}")
    return buffer

def save_img(fname, tensor):
    img_tensor = np.squeeze(tensor,axis=2)  # Remove batch dim → [C, H, W]
    img = Image.fromarray(img_tensor.astype(np.uint8), mode='L')  # 'L' = 8-bit pixels, black and white
    save_path=f"{fname}.png"
    img.save(save_path)
    return save_path

def unpack_frame(packed):
    H, W_packed,ch = packed.shape  # (480, 160,4)
    W = W_packed * ch            # Unpacked width = 640
    output_tensor = np.reshape(frame, (H, W, 1))
    return output_tensor

def save_frame(sufix, frame):
    print(f"type(frame)={type(frame)},  \
          frame.shape={frame.shape},frame.dtype={frame.dtype}")
    unpacked_frame  = frame.view(dtype=np.uint8).reshape((480, 640, 1))
    print(f"type(unpacked_frame)={type(unpacked_frame)},\nunpacked_frame.shape={unpacked_frame.shape}, \
          \nunpacked_frame.dtype={unpacked_frame.dtype}")
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    #fname=f"{timestamp}-{sufix}"
    fname=f"image-output-{sufix}"
    save_img(fname=fname, tensor=unpacked_frame)

In [5]:
img_fname1='test_image.jpg'
img_fname2='color_bars_1920x1080.png'

In [6]:
buff_1=image_to_RGB(img_fname1)
buff_2=image_to_RGB(img_fname2)
cnt=0

Packed buffer shape: (1080, 1920), dtype: uint32
Packed buffer shape: (1080, 1920), dtype: uint32


In [7]:
# ----------------------------------------
# VIVADO configuration
# ----------------------------------------
# VDMA
VDMA_BASE_ADDR = 0xA001_0000  # Replace with actual base address
VDMA_RANGE     = 0x0_8000    # 32K
FRAME_BUFFERS  = 2  #buggy
#FRAME_BUFFERS  = 1  
# ----------------------------------------
# FRAME configuration
# ----------------------------------------
WIDTH          = 160
HEIGHT         = 480
BPP            = 4           # Bytes per pixel (32bpp)
STRIDE         = WIDTH * BPP # Bytes per line
FRAME_SIZE     = STRIDE * HEIGHT

# ----------------------------------------
# IMG2AXIS
# ----------------------------------------
IMG2AXIS_BASE_ADDR = 0xA001_8000
IMG2AXIS_RANGE =0x0_8000

In [8]:
from pynq import MMIO
img2axis_mmio = MMIO(IMG2AXIS_BASE_ADDR, IMG2AXIS_RANGE)
# ----------------------------------------
vdma_mmio = MMIO(VDMA_BASE_ADDR, VDMA_RANGE)

In [16]:

#just configure hls, no start
cfg_img_to_axis(img2axis_mmio,buff_1 if cnt % 2 else buff_2,0,8)
cnt+=1
#start AXIS in a loop
#img_to_axis(img2axis_mmio,buff_2,0,8,True)

In [17]:
dump_hls_ctrl(img2axis_mmio)

----- hls Dump -----
Control signals                  0x00: 0x0000000E
Global Interrupt Enable Register 0x04: 0x00000000
IP Interrupt Enable Register     0x08: 0x00000000
IP Interrupt Status Register     0x0C: 0x00000000
Control Flags:
 - Bit 1: ap_done (Read/COR)
 - Bit 2: ap_idle (Read)
 - Bit 3: ap_ready (Read/COR)
----------------------------------


In [18]:
dump_s2mm_status(vdma_mmio)

----- VDMA S2MM Status Dump -----
Control Reg     (0x30): 0x00010001
Status Reg      (0x34): 0x00011000
Vertical Size   (0xA0): 480
Horizontal Size (0xA4): 640
Stride          (0xA8): 640
S2MM_SA1        (0xAC): 0x60080000
S2MM_SA2        (0xB0): 0x60280000
Status Flags:
 - Bit 12: Frame Count Interrupt
----------------------------------


# ----------------------------------------------------------------------------------------------------------

In [12]:
vdma_mmio.write(VDMA_S2MM["S2MM_VDMACR"], 0x00000004)  # Reset
#640,480,60080000,60280000
vdma_mmio.write(VDMA_S2MM["S2MM_SA1"], 0x60080000)
vdma_mmio.write(VDMA_S2MM["S2MM_SA2"], 0x60280000)

# ----------------------------------------
# Set stride (bytes per row)
# ----------------------------------------
vdma_mmio.write(VDMA_S2MM["S2MM_STRIDE"], 640)

# ----------------------------------------
# Set horizontal size (in bytes)
# ----------------------------------------
vdma_mmio.write(VDMA_S2MM["S2MM_HSIZE"], 640)

In [13]:

dump_s2mm_status(vdma_mmio)

----- VDMA S2MM Status Dump -----
Control Reg     (0x30): 0x00010002
Status Reg      (0x34): 0x00000000
Vertical Size   (0xA0): 0
Horizontal Size (0xA4): 0
Stride          (0xA8): 0
S2MM_SA1        (0xAC): 0x00000000
S2MM_SA2        (0xB0): 0x00000000
Status Flags:
----------------------------------
